In [3]:
model_name = "distilgpt2"
device = "cpu"

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model= AutoModelForCausalLM.from_pretrained(model_name)

model

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 847.72it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
import pandas as pd
df = pd.read_csv("./movie_plots.csv")
df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [6]:
df=df[:1000]
df

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...
...,...,...,...,...,...,...,...,...
995,1930,Playing Around,American,Mervyn LeRoy,"Alice White, Chester Morris",drama,https://en.wikipedia.org/wiki/Playing_Around,Alice White plays the part of a working class ...
996,1930,Raffles,American,George Fitzmaurice,"Ronald Colman, Kay Francis",mystery,https://en.wikipedia.org/wiki/Raffles_(1930_film),Gentleman jewel thief A.J. Raffles (Ronald Col...
997,1930,Reaching for the Moon,American,Edmund Goulding,"Douglas Fairbanks, Edward Everett Horton, Bing...",musical,https://en.wikipedia.org/wiki/Reaching_for_the...,"Wall Street wizard, Larry Day, new to the ways..."
998,1930,Recaptured Love,American,John G. Adolfi,"Belle Bennett, Dorothy Burgess",musical,https://en.wikipedia.org/wiki/Recaptured_Love,"In this drama, a 50-year-old married man (play..."


In [7]:
from datasets import Dataset

def map_dataset(batch):
  return tokenizer(
    batch["Plot"],
    truncation=True,
    max_length=128,
    return_overflowing_tokens=True
    )
  

dataset= Dataset.from_pandas(df)
dataset=dataset.map(
  map_dataset,
  batched = True,
  batch_size =8,
  remove_columns = list(df.columns)
)


dataset=dataset.remove_columns(["overflow_to_sample_mapping"])
dataset= dataset.train_test_split(test_size=0.2)
dataset


Map: 100%|██████████| 1000/1000 [00:01<00:00, 520.69 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2188
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 548
    })
})

In [8]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
  tokenizer=tokenizer,
  mlm=False
)

data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensors='pt', seed=None)

In [9]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
  output_dir="./output/model",
  eval_strategy="epoch",
  # CPU-Specific Optimizations
  no_cuda=True,
  dataloader_pin_memory=False,   # Turns off the flag causing your warning
  per_device_train_batch_size=4,
  use_cpu=True,
  learning_rate=2e-5,
  weight_decay=0.01,
  num_train_epochs=10
)

trainer = Trainer(
  model=model,
  train_dataset=dataset["train"],
  eval_dataset=dataset["test"],
  args=training_args,
  data_collator=data_collator
)

trainer.train()

c:\Users\Joe\Documents\AI_training\movie_llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.759562
2,3.899060,3.717349
3,3.899060,3.703426
4,3.669456,3.699846
5,3.669456,3.693934
6,3.577797,3.694934
7,3.577797,3.696857
8,3.517961,3.698408
9,3.517961,3.697955
10,3.480146,3.698246


Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.41s/it]
c:\Users\Joe\Documents\AI_training\movie_llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.77s/it]
c:\Users\Joe\Documents\AI_training\movie_llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.19s/it]
c:\Users\Joe\Documents\AI_training\movie_llm\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

TrainOutput(global_step=2740, training_loss=3.6143391407319228, metrics={'train_runtime': 25480.2505, 'train_samples_per_second': 0.859, 'train_steps_per_second': 0.108, 'total_flos': 714564957634560.0, 'train_loss': 3.6143391407319228, 'epoch': 10.0})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
# Tell the tokenizer to use the EOS token for padding
tokenizer.pad_token = tokenizer.eos_token
model= AutoModelForCausalLM.from_pretrained("./output/model/checkpoint-2740")

prompt = "A short film about a pilot who"

#Tokenize the prompt (this now creates an accurate attention_mask)
inputs = tokenizer(prompt,return_tensors="pt").to("cpu")

# Pass **inputs (unpacks both input_ids AND attention_mask) 
# and explicitly set the pad_token_id

outputs = model.generate(
  #inputs.input_ids,
  **inputs,
  max_new_tokens=100,
  do_sample=True,
  top_k=50,
  top_p=.95
)

output_string=tokenizer.batch_decode(outputs)

output_string

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 3802.13it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


["A short film about a pilot who's saved a life. In an attempt to protect the pilot from danger, he's killed two pilots. During the film, his two pilots are reunited with his wife's beautiful granddaughter and is saved by the pilot's father. Before the crash, a rescue aircraft arrives. The pilot and his crew are rescued when a parachute is released from the aircraft's parachute, causing a hole in the wreckage and causing damage to the aircraft's cabin. At that moment, the pilot's daughter and granddaughter are able to"]